# CUDA-Q hybrid CPU–QPU playground

This notebook uses CUDA-Q's **`qpp-cpu` simulator** as a local QPU. Ordinary Python performs the classical work, while CUDA-Q compiles and executes the quantum kernels. No cloud account or physical QPU is required.

In [ ]:
import platform

import cudaq
import matplotlib.pyplot as plt
import numpy as np

cudaq.set_target("qpp-cpu")
print(f"CUDA-Q version: {cudaq.__version__}")
print(f"Python platform: {platform.platform()}")
print(f"Simulation target: {cudaq.get_target().name}")

## 1. Build and sample a Bell state

The quantum kernel creates $\left(|00\rangle + |11\rangle\right)/\sqrt{2}$. A correct run should therefore contain only `00` and `11`, with roughly equal counts.

In [ ]:
@cudaq.kernel
def bell_state():
    qubits = cudaq.qvector(2)
    h(qubits[0])
    x.ctrl(qubits[0], qubits[1])
    mz(qubits)

print(cudaq.draw(bell_state))

In [ ]:
cudaq.set_random_seed(7)
counts = cudaq.sample(bell_state, shots_count=1_000)
print(counts)

observed = {bits for bits, _ in counts.items()}
assert observed <= {"00", "11"}

## 2. Hybrid CPU–QPU variational program

Here the **CPU** proposes an angle $\theta$. The simulated **QPU** prepares a parameterized state and evaluates $\langle H \rangle$. Repeating that exchange is the core pattern behind algorithms such as the Variational Quantum Eigensolver (VQE).

In [ ]:
from cudaq import spin

hamiltonian = (
    5.907
    - 2.1433 * spin.x(0) * spin.x(1)
    - 2.1433 * spin.y(0) * spin.y(1)
    + 0.21829 * spin.z(0)
    - 6.125 * spin.z(1)
)

@cudaq.kernel
def ansatz(theta: float):
    qubits = cudaq.qvector(2)
    x(qubits[0])
    ry(theta, qubits[1])
    x.ctrl(qubits[1], qubits[0])

def energy(theta: float) -> float:
    # This call crosses from classical Python into the QPU simulator.
    return float(cudaq.observe(ansatz, hamiltonian, theta).expectation())

print(cudaq.draw(ansatz, 0.25))
print(f"Energy at theta=0.25: {energy(0.25):.8f}")

In [ ]:
# Classical CPU search; each energy evaluation executes the quantum kernel.
thetas = np.linspace(-np.pi, np.pi, 81)
energies = np.array([energy(float(theta)) for theta in thetas])
best_index = int(np.argmin(energies))
best_theta = float(thetas[best_index])
best_energy = float(energies[best_index])

print(f"Best grid angle: {best_theta:.6f} rad")
print(f"Best grid energy: {best_energy:.8f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thetas, energies, label=r"$\langle H \rangle$")
ax.scatter([best_theta], [best_energy], color="crimson", zorder=3, label="best grid point")
ax.set(xlabel=r"$\theta$ (radians)", ylabel="Energy", title="Hybrid variational energy landscape")
ax.grid(alpha=0.25)
ax.legend();

## 3. Refine the answer with a classical feedback loop

This deliberately small optimizer makes the hybrid boundary visible: Python compares candidate energies, while every candidate is evaluated by CUDA-Q.

In [ ]:
theta = best_theta
history = []

for step in (0.1, 0.03, 0.01, 0.003, 0.001):
    candidates = (theta - step, theta, theta + step)
    candidate_energies = [energy(value) for value in candidates]
    choice = int(np.argmin(candidate_energies))
    theta = candidates[choice]
    history.append((step, theta, candidate_energies[choice]))

for step, value, result in history:
    print(f"step={step:>5g}  theta={value: .6f}  energy={result: .9f}")

print(f"\nOptimized theta: {theta:.6f} rad")
print(f"Optimized energy: {energy(theta):.9f}")

## Things to try next

- Change the Hamiltonian coefficients and watch the optimum move.
- Pass a positive `shots_count` to `cudaq.observe(...)` to see sampling noise.
- Increase the number of qubits and compare execution time.
- On a supported NVIDIA GPU visible inside WSL2, try `cudaq.set_target("nvidia")`.